In [1]:
pip install PyMuPDF

   ---------------------------------------- 0.0/18.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.4 MB ? eta -:--:--
    --------------------------------------- 0.3/18.4 MB ? eta -:--:--
   -- ------------------------------------- 1.0/18.4 MB 3.2 MB/s eta 0:00:06
   ------- -------------------------------- 3.4/18.4 MB 6.7 MB/s eta 0:00:03
   -------------- ------------------------- 6.6/18.4 MB 8.9 MB/s eta 0:00:02
   -------------------- ------------------- 9.4/18.4 MB 10.0 MB/s eta 0:00:01
   ------------------------- -------------- 11.5/18.4 MB 10.1 MB/s eta 0:00:01
   ------------------------------- -------- 14.4/18.4 MB 10.6 MB/s eta 0:00:01
   ------------------------------------- -- 17.0/18.4 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------  18.1/18.4 MB 10.5 MB/s eta 0:00:01
   ---------------------------------------  18.1/18.4 MB 10.5 MB/s eta 0:00:01
   ---------------------------------------  18.1/18.4 MB 10.5 MB/s eta 0:00:01
   ----


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# STEP 1 — Extract text from the quarterly report PDF

import fitz  # PyMuPDF

pdf_path = "HBL_Quarterly_Report_September_30,_2024.pdf"

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text() + "\n"
    return text

raw_text = extract_text_from_pdf(pdf_path)

print(raw_text[:2000])  # show first 2000 chars to verify extraction


Enriching Lives
A n n u a l  R e p o r t 2023
Quarterly Report
September 30, 2024

Corporate Information
2
Condensed Interim Consolidated 
Financial Statements
Directors’ Review - English
3
Directors’ Review - Urdu
6
Condensed Interim Consolidated Statement of 
Financial Position
12
Condensed Interim Consolidated Profit and 
Loss Account 
13
Condensed Interim Consolidated Statement of 
Comprehensive Income 
14
Condensed Interim Consolidated Statement of 
Changes in Equity 
15
Condensed Interim Consolidated Cash Flow 
Statement 
16
Notes to the Condensed Interim Consolidated 
Financial Statements
17
Condensed Interim Unconsolidated 
Financial Statements
Directors’ Review - English
52
Directors’ Review - Urdu
55
Condensed Interim Unconsolidated Statement 
of Financial Position 
59
Condensed Interim Unconsolidated Profit and 
Loss Account 
60
Condensed Interim Unconsolidated Statement 
of Comprehensive Income 
61
Condensed Interim Unconsolidated Statement 
of Changes in Equity 
62
Condens

In [4]:
# STEP 2 — Clean Urdu text, remove tables, noise, and non-content

import re

def clean_financial_report(text):

    # -----------------------------------------
    # 1) Remove Urdu (Arabic script characters)
    # -----------------------------------------
    text = re.sub(r'[\u0600-\u06FF]+', ' ', text)

    # -------------------------------------------------
    # 2) Remove emails, websites, phone numbers
    # -------------------------------------------------
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\+?\d[\d\s\-\(\)]{6,}', ' ', text)

    # -------------------------------------------------
    # 3) Remove numeric table rows (common in PDFs)
    # Example: 12,300  45,200  (200)  92,300
    # -------------------------------------------------
    text = re.sub(r'[\d,\.]+(?:\s+[\d,\.]+){2,}', ' ', text)

    # -------------------------------------------------
    # 4) Remove page numbers, footers, headers
    # -------------------------------------------------
    text = re.sub(r'\bPage\s*\d+\b', ' ', text)
    text = re.sub(r'\b\d{1,3}\b', ' ', text)  # standalone small numbers
    text = re.sub(r'HABIB BANK LIMITED|HBL|Habib Bank Limited', ' ', text, flags=re.I)

    # -------------------------------------------------
    # 5) Remove short lines (addresses, random junk)
    # We keep only meaningful sentences/paragraphs
    # -------------------------------------------------
    cleaned_lines = []
    for line in text.split("\n"):
        line = line.strip()
        if len(line) > 40:        # keep only long lines
            cleaned_lines.append(line)
    text = " ".join(cleaned_lines)

    # -------------------------------------------------
    # 6) Remove repeated spaces
    # -------------------------------------------------
    text = re.sub(r'\s+', ' ', text)

    return text


cleaned_text = clean_financial_report(raw_text)

print(cleaned_text[:2000])   # preview first cleaned chunk


Condensed Interim Consolidated Statement of Condensed Interim Consolidated Profit and Condensed Interim Consolidated Statement of Condensed Interim Consolidated Statement of Notes to the Condensed Interim Consolidated Condensed Interim Unconsolidated Statement Condensed Interim Unconsolidated Profit and Condensed Interim Unconsolidated Statement Condensed Interim Unconsolidated Statement Condensed Interim Unconsolidated Cash Flow Notes to the Condensed Interim Unconsolidated With a . % GDP growth in FY’ , Pakistan’s nascent economic recovery continues, underpinned by favourable global commodity prices, improved fiscal discipline and a more stable external position. The Large-Scale Manufacturing Index is regaining footing, with the Jul’ reading exhibiting a growth of . %. High frequency sales indicators and increased capacity utilization also reflect a moderate pickup in industrial activity. Inflation of . % for Aug’ was the first single digit reading since Oct’ and dropped further to a

In [6]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [7]:
# STEP 3 — Split into sentences and filter only useful narrative content

import re
import nltk
nltk.download('punkt')

# ---------------------------------
# 3.1) Sentence tokenization
# ---------------------------------
sentences = nltk.sent_tokenize(cleaned_text)

print("Total sentences extracted:", len(sentences))
print("\nSample:", sentences[:5])


# ----------------------------------------------------------
# 3.2) Financial narrative section keywords
# These help us filter only relevant analysis text
# ----------------------------------------------------------
financial_sections_keywords = [
    "macroeconomic", "economic", "gdp", "inflation", "policy rate",
    "business", "performance", "overview", "income", "profit", "earnings",
    "capital", "liquidity", "risk", "credit", "outlook", "future",
    "market", "deposit", "advances", "loans", "assets",
    "digital", "technology", "growth", "strategy", "challenges"
]

# ----------------------------------------------------------
# 3.3) Filter sentences that contain these keywords
# ----------------------------------------------------------
filtered_sentences = []
for s in sentences:
    s_low = s.lower()
    if any(kw in s_low for kw in financial_sections_keywords):
        filtered_sentences.append(s)

print("\nFiltered narrative sentences:", len(filtered_sentences))
print("\nSample filtered:", filtered_sentences[:5])


Total sentences extracted: 707

Sample: ['Condensed Interim Consolidated Statement of Condensed Interim Consolidated Profit and Condensed Interim Consolidated Statement of Condensed Interim Consolidated Statement of Notes to the Condensed Interim Consolidated Condensed Interim Unconsolidated Statement Condensed Interim Unconsolidated Profit and Condensed Interim Unconsolidated Statement Condensed Interim Unconsolidated Statement Condensed Interim Unconsolidated Cash Flow Notes to the Condensed Interim Unconsolidated With a .', '% GDP growth in FY’ , Pakistan’s nascent economic recovery continues, underpinned by favourable global commodity prices, improved fiscal discipline and a more stable external position.', 'The Large-Scale Manufacturing Index is regaining footing, with the Jul’ reading exhibiting a growth of .', '%.', 'High frequency sales indicators and increased capacity utilization also reflect a moderate pickup in industrial activity.']

Filtered narrative sentences: 245

Samp

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [8]:
# STEP 4 — Define financial aspects + keywords for ABSA

aspects_keywords = {
    "profit": [
        "profit", "earning", "earnings", "income", "net interest income",
        "profit after tax", "eps", "revenue", "topline"
    ],

    "deposits": [
        "deposit", "deposits", "casa", "current accounts",
        "savings accounts", "low-cost deposits"
    ],

    "advances": [
        "advance", "advances", "lending", "loan", "loans", "asset growth"
    ],

    "capital": [
        "capital", "capital adequacy", "car", "tier 1", "ceti",
        "buffers", "capital ratio", "reserves"
    ],

    "npl": [
        "non-performing", "npl", "non performing loans",
        "infection ratio", "non-performing portfolio"
    ],

    "provisions": [
        "provision", "provisioning", "credit loss allowance",
        "impairment", "coverage"
    ],

    "macroeconomic": [
        "inflation", "gdp", "growth", "commodity prices",
        "macroeconomic", "trade deficit", "remittances",
        "policy rate", "sbp", "monetary policy", "imf"
    ],

    "rating": [
        "credit rating", "moodys", "fitch", "outlook revised",
        "stable outlook", "rating agency"
    ],

    "outlook": [
        "outlook", "future", "expected", "guidance", "forecast",
        "trajectory", "prospects"
    ],

    "risk": [
        "risk", "liquidity risk", "market risk", "credit risk",
        "operational risk", "stress", "challenge", "volatility"
    ],

    "digital": [
        "digital", "mobile banking", "internet banking", "pos",
        "cards", "digitalization", "technology"
    ],

    "costs": [
        "cost", "expense", "operating expenses", "cost-to-income",
        "administrative expenses"
    ]
}

print("Aspects defined:", len(aspects_keywords))


Aspects defined: 12


In [9]:
# STEP 5 — Perform Aspect Matching + Sentiment Scoring for ABSA

import pandas as pd
import re

# --------------------------------------------
# 5.1) Finance-specific positive/negative words
# --------------------------------------------

positive_words = set("""
increase increased improving improved growth growing strengthen strengthened
robust strong favourable positive rise rising expansion boost recovering
""".split())

negative_words = set("""
decrease decreased decline declined weak weakening fall falling drop dropped
challenge challenging risks stress stressed inflation spike contraction slowdown
""".split())

# --------------------------------------------
# 5.2) Sentiment Scoring Function (rule-based)
# --------------------------------------------

def score_sentence(sent):
    """Return sentiment: +1, 0, -1"""
    s = re.sub(r'[^a-zA-Z0-9\s]', ' ', sent.lower())  # clean punctuation
    tokens = s.split()

    pos = sum(1 for t in tokens if t in positive_words)
    neg = sum(1 for t in tokens if t in negative_words)

    if pos == 0 and neg == 0:
        return 0    # neutral if no sentiment words

    score = pos - neg

    # Convert to coarse score
    if score > 0:
        return 1
    elif score < 0:
        return -1
    return 0


# --------------------------------------------
# 5.3) Match each sentence to an aspect
# --------------------------------------------

absa_rows = []

for sent in filtered_sentences:
    sent_low = sent.lower()
    matched_any = False

    for aspect, keywords in aspects_keywords.items():
        if any(kw in sent_low for kw in keywords):
            sentiment = score_sentence(sent)
            absa_rows.append({
                "sentence": sent,
                "aspect": aspect,
                "sentiment": sentiment
            })
            matched_any = True

    # Optional: if sentence matched no aspect → ignore it

# --------------------------------------------
# 5.4) Create ABSA dataframe
# --------------------------------------------

absa_df = pd.DataFrame(absa_rows)

print("Total ABSA sentences:", len(absa_df))
absa_df.head(10)


Total ABSA sentences: 432


,sentence,aspect,sentiment
0,Condensed Interim Consolidated Statement of Co...,profit,0
1,"% GDP growth in FY’ , Pakistan’s nascent econo...",macroeconomic,1
2,"% GDP growth in FY’ , Pakistan’s nascent econo...",digital,1
3,The Large-Scale Manufacturing Index is regaini...,macroeconomic,1
4,Inflation of .,macroeconomic,-1
5,Average inflation fell from .,macroeconomic,-1
6,% due to robust agricultural growth.,macroeconomic,1
7,"Strong remittance growth continued in FY’ , wi...",macroeconomic,1
8,Key priorities under the program are i) streng...,macroeconomic,1
9,Key priorities under the program are i) streng...,costs,1


In [10]:
# STEP 6 — Aggregate ABSA aspect scores & compute WASI

import pandas as pd

# -------------------------------------------------------
# 6.1) Aggregate aspect sentiment from absa_df
# -------------------------------------------------------

aspect_summary = (
    absa_df.groupby("aspect")
           .agg(
               count=("sentiment", "count"),
               mean_sentiment=("sentiment", "mean"),
               positive=("sentiment", lambda s: (s == 1).sum()),
               negative=("sentiment", lambda s: (s == -1).sum())
           )
           .reset_index()
)

print("Aspect Summary Table:")
display(aspect_summary)


# -------------------------------------------------------
# 6.2) Define aspect weights (based on banking importance)
# -------------------------------------------------------

aspect_weights = {
    "profit": 0.20,
    "deposits": 0.15,
    "advances": 0.15,
    "capital": 0.08,
    "npl": 0.08,
    "provisions": 0.05,
    "macroeconomic": 0.10,
    "rating": 0.05,
    "outlook": 0.10,
    "risk": 0.02,
    "digital": 0.01,
    "costs": 0.01
}

# normalize weights to sum to 1
total_w = sum(aspect_weights.values())
aspect_weights = {k: v / total_w for k, v in aspect_weights.items()}

print("\nNormalized Weights:")
aspect_weights


# -------------------------------------------------------
# 6.3) Compute WASI (Weighted Aspect Sentiment Index)
# -------------------------------------------------------

wasi_value = 0

for idx, row in aspect_summary.iterrows():
    aspect = row["aspect"]
    score = row["mean_sentiment"]

    weight = aspect_weights.get(aspect, 0)

    wasi_value += score * weight

print("\nFinal Weighted Aspect Sentiment Index (WASI):", wasi_value)


# -------------------------------------------------------
# 6.4) Create a final sentiment features table
# -------------------------------------------------------

wasi_df = aspect_summary.copy()
wasi_df["weight"] = wasi_df["aspect"].map(aspect_weights)
wasi_df["weighted_score"] = wasi_df["mean_sentiment"] * wasi_df["weight"]

print("\nWASI Feature Table:")
display(wasi_df)



Aspect Summary Table:


,aspect,count,mean_sentiment,positive,negative
0,advances,29,0.000000,2,2
1,capital,43,0.348837,15,0
2,costs,35,0.114286,6,2
3,deposits,33,0.242424,8,0
4,digital,68,0.323529,22,0
5,macroeconomic,67,0.373134,33,8
6,npl,2,0.000000,0,0
7,outlook,21,0.523810,13,2
8,profit,79,0.113924,14,5
9,provisions,36,0.138889,5,0



Normalized Weights:

Final Weighted Aspect Sentiment Index (WASI): 0.2404253453325972

WASI Feature Table:


,aspect,count,mean_sentiment,positive,negative,weight,weighted_score
0,advances,29,0.000000,2,2,0.15,0.000000
1,capital,43,0.348837,15,0,0.08,0.027907
2,costs,35,0.114286,6,2,0.01,0.001143
3,deposits,33,0.242424,8,0,0.15,0.036364
4,digital,68,0.323529,22,0,0.01,0.003235
5,macroeconomic,67,0.373134,33,8,0.10,0.037313
6,npl,2,0.000000,0,0,0.08,0.000000
7,outlook,21,0.523810,13,2,0.10,0.052381
8,profit,79,0.113924,14,5,0.20,0.022785
9,provisions,36,0.138889,5,0,0.05,0.006944
